# Funcs

In [1]:
import json
import os

def read_json(file_path):
    """Reads a JSON file and returns the data."""
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)
    return data

def save_json(data, file_path, indent=4):
    """Saves data to a JSON file."""
    with open(file_path, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=indent)

def read_html_files(folder_path):
    """Reads all HTML files in a folder and returns a dictionary with filenames as keys and file contents as values."""
    html_dict = {}

    for file_name in os.listdir(folder_path):
        if file_name.endswith(".html"):
            file_path = os.path.join(folder_path, file_name)
            year = int(file_name[:-5].split('_')[0])
            id_in_year = int(file_name[:-5].split('_')[1])
            with open(file_path, "r", encoding="utf-8") as file:
                html_dict[(year, id_in_year)] = file.read()
    
    return html_dict

# Merging

In [2]:
semi_cleaned = read_json('semi_cleaned_docs.json')
the_rest = read_json('the_rest_updated 1.json')

In [3]:
len(semi_cleaned)

132

In [4]:
len(the_rest)

38

In [5]:
merged = semi_cleaned + the_rest

for merged_doc in merged:
    print(merged_doc['id'])

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170


In [6]:
save_json(merged, 'final_merged.json')

# Extracting Gold pieces

In [7]:
merged_ = read_json('final_merged.json')

In [8]:
sample = merged_[0:1]

In [9]:
sample[0]['data']['html']

'<document>\n<type>EX-10.1</type>\n<sequence>2</sequence>\n<filename>exhibit101creditagreement.htm</filename>\n<description>EX-10.1</description>\n<text>\n<!DOCTYPE html PUBLIC "-//W3C//DTD HTML 4.01 Transitional//EN" "http://www.w3.org/TR/html4/loose.dtd">\n\n<!-- Document created using Wdesk -->\n<!-- Copyright 2022 Workiva -->\n<title>Document</title><div id="i96ac4f6ed0ce43f083deb4a266dd81e8_1"></div><div style="min-height:72pt;width:100%"><div style="margin-bottom:0.12pt;text-align:right;text-indent:0.36pt"><font style="color:#000000;font-family:\'Times New Roman\',sans-serif;font-size:11pt;font-weight:700;line-height:100%">EXHIBIT 10.1</font></div></div><div style="margin-bottom:0.12pt;text-align:justify;text-indent:0.36pt"><table style="border-collapse:collapse;display:inline-table;margin-bottom:5pt;vertical-align:text-bottom;width:100.000%"><tr><td style="width:1.0%"></td><td style="width:98.900%"></td><td style="width:0.1%"></td></tr><tr style="height:32pt"><td colspan="3" sty

In [10]:
sample[0]['annotations'][0]

{'id': 1,
 'completed_by': 1,
 'result': [{'value': {'start': '/document[1]/text[1]/div[886]/font[1]/text()[1]',
    'startOffset': 0,
    'end': '/document[1]/text[1]/div[886]/font[1]/text()[1]',
    'endOffset': 24,
    'globalOffsets': {'start': 305599, 'end': 305623},
    'text': 'FRANKLIN RESOURCES, INC.',
    'hypertextlabels': ['Organization Name']},
   'id': 'aaaaaaaaaa',
   'from_name': 'ner',
   'to_name': 'text',
   'type': 'hypertextlabels',
   'origin': 'manual'},
  {'value': {'start': '/document[1]/text[1]/div[890]/font[1]/text()[1]',
    'startOffset': 7,
    'end': '/document[1]/text[1]/div[890]/font[1]/text()[1]',
    'endOffset': 20,
    'globalOffsets': {'start': 305669, 'end': 305682},
    'text': 'Mark Constant',
    'hypertextlabels': ['Person Name']},
   'id': 'aaaaaaaaab',
   'from_name': 'ner',
   'to_name': 'text',
   'type': 'hypertextlabels',
   'origin': 'manual'},
  {'value': {'start': '/document[1]/text[1]/div[891]/font[1]/text()[1]',
    'startOffset': 8

In [11]:
save_json(sample, 'sample.json')

In [12]:
import json
from bs4 import BeautifulSoup

def extract_and_merge_snippets_with_entities(data, padding=100):
    raw_html = data[0]["data"]["html"]
    annotations = data[0]["annotations"][0]["result"]
    soup = BeautifulSoup(raw_html, 'html.parser')
    clean_text = soup.get_text()

    raw_spans = []

    # Step 1: Get all matched annotation spans
    for ann in annotations:
        value = ann.get("value", {})
        if "text" not in value or "hypertextlabels" not in value:
            continue
        entity_text = value["text"].strip()
        label = value["hypertextlabels"]

        start_idx = 0
        while True:
            idx = clean_text.find(entity_text, start_idx)
            if idx == -1:
                break
            span_start = max(0, idx - padding)
            span_end = min(len(clean_text), idx + len(entity_text) + padding)
            raw_spans.append({
                "start": span_start,
                "end": span_end,
                "entity_start": idx,
                "entity_end": idx + len(entity_text),
                "entity_text": entity_text,
                "label": label
            })
            start_idx = idx + len(entity_text)

    # Step 2: Sort and merge overlapping spans
    if not raw_spans:
        return []

    # sort by span start
    raw_spans.sort(key=lambda x: x["start"])
    merged_snippets = []
    current = {
        "start": raw_spans[0]["start"],
        "end": raw_spans[0]["end"],
        "entities": [raw_spans[0]]
    }

    for span in raw_spans[1:]:
        if span["start"] <= current["end"]:  # overlap
            current["end"] = max(current["end"], span["end"])
            current["entities"].append(span)
        else:
            merged_snippets.append(current)
            current = {
                "start": span["start"],
                "end": span["end"],
                "entities": [span]
            }
    merged_snippets.append(current)

    # Step 3: Final output
    results = []
    for snippet in merged_snippets:
        snippet_text = clean_text[snippet["start"]:snippet["end"]]
        matched_entities = list({
            e["entity_text"] for e in snippet["entities"]
            if snippet["start"] <= e["entity_start"] < snippet["end"]
        })

        results.append({
            "context_snippet": snippet_text,
            "matched_entities": matched_entities
        })

    return results

# Example usage
snippets = extract_and_merge_snippets_with_entities(sample, padding=100)
for i, s in enumerate(snippets):
    print(f"\nSnippet {i+1}:")
    print("Matched Entities:", s["matched_entities"])
    print("Context:\n", s["context_snippet"])
    print('-' * 50)


Snippet 1:
Matched Entities: ['Administrative Agent', 'Lender', 'Lenders', 'BANK OF AMERICA, N.A.', 'FRANKLIN RESOURCES, INC.', 'Arranger', 'Borrower', 'Bookrunner', 'Lead']
Context:
 reditagreement.htm
EX-10.1




DocumentEXHIBIT 10.1CREDIT AGREEMENTdated as of January 10, 2022amongFRANKLIN RESOURCES, INC.,as the Borrower,BANK OF AMERICA, N.A.,as Administrative Agent,andthe Lenders party heretoBANK OF AMERICA, N.A.,asSole Lead Arranger and Sole Bookrunner                TABLE OF CONTENTSSectionPageARTICLE IDEFINITIONS AND ACCOUNTING TERMS11.01Defined Te
--------------------------------------------------

Snippet 2:
Matched Entities: ['Administrative Agent', 'Lender', 'Lenders']
Context:
 Commitment Fee252.10Computation of Interest and Fees262.11Evidence of Debt262.12Payments Generally; Administrative Agent’s Clawback.262.13Sharing of Payments by Lenders282.14[Reserved].282.15[Reserved].282.16[Reserved].282.17Defaulting Lenders.282.18Defaulting Lender Cure29ARTICLE IIITAXES, YIELD PRO

In [8]:
import json
from bs4 import BeautifulSoup

def extract_merged_snippets_per_doc(data, padding=100):
    merged_by_doc = []

    for doc in data:
        raw_html = doc["data"]["html"]
        annotations = doc["annotations"][0]["result"]
        soup = BeautifulSoup(raw_html, 'html.parser')
        clean_text = soup.get_text()

        raw_spans = []

        for ann in annotations:
            value = ann.get("value", {})
            if "text" not in value:
                continue
            entity_text = value["text"].strip()

            start_idx = 0
            while True:
                idx = clean_text.find(entity_text, start_idx)
                if idx == -1:
                    break
                span_start = max(0, idx - padding)
                span_end = min(len(clean_text), idx + len(entity_text) + padding)
                raw_spans.append((span_start, span_end))
                start_idx = idx + len(entity_text)

        if not raw_spans:
            merged_by_doc.append("")  # empty string for this doc
            continue

        raw_spans.sort()
        merged = [raw_spans[0]]
        for start, end in raw_spans[1:]:
            last_start, last_end = merged[-1]
            if start <= last_end:
                merged[-1] = (last_start, max(last_end, end))
            else:
                merged.append((start, end))

        merged_text = ""
        for start, end in merged:
            merged_text += clean_text[start:end] + "\n\n"

        merged_by_doc.append(merged_text.strip())
        print(f"\n=== Document {doc['id']} ===")
        print(f'Doc length: {len(doc["data"]["html"])}')
        print(f'Text length: {len(merged_by_doc[-1])}')
        print(merged_by_doc[-1][:500])  # Preview first 500 chars per doc
        print("-" * 50)

    return merged_by_doc

# Usage
merged_per_doc = extract_merged_snippets_per_doc(merged_, padding=100)

for i, text in enumerate(merged_per_doc):
    print(f"\n=== Document {i + 1} ===")
    print(f'Doc length: {len(merged_[i]["data"]["html"])}')
    print(f'Text length: {len(text)}')
    print(text[:500])  # Preview first 500 chars per doc
    print("-" * 50)


=== Document 1 ===
Doc length: 828619
Text length: 174998
reditagreement.htm
EX-10.1




DocumentEXHIBIT 10.1CREDIT AGREEMENTdated as of January 10, 2022amongFRANKLIN RESOURCES, INC.,as the Borrower,BANK OF AMERICA, N.A.,as Administrative Agent,andthe Lenders party heretoBANK OF AMERICA, N.A.,asSole Lead Arranger and Sole Bookrunner                TABLE OF CONTENTSSectionPageARTICLE IDEFINITIONS AND ACCOUNTING TERMS11.01Defined Te

Commitment Fee252.10Computation of Interest and Fees262.11Evidence of Debt262.12Payments Generally; Administrative Agent’s
--------------------------------------------------

=== Document 2 ===
Doc length: 607168
Text length: 226010
10.1

Exhibit 10.1 
EXECUTION VERSION   
   
CREDIT AGREEMENT  DATED
AS OF JANUARY 14, 2022  AMONG 
DICKS SPORTING GOODS, INC., 
as the Borrower,  THE
LENDERS FROM TIME TO TIME PARTIES HERETO,  and 
WELLS FARGO BANK, NATIONAL ASSOCIATION, 
as Administrative Agent and an Issuing Lender 
WELLS FARGO SECURITIES, LLC, 
and  BOFA SE

KeyboardInterrupt: 

In [35]:
import json
import re
from bs4 import BeautifulSoup

def extract_clean_text_snippets(data, padding=100):
    snippets = {}
    
    for i in range(len(data)):
        snippets[data[i]['id']] = []
        
        raw_html = data[i]["data"]["html"]
        annotations = data[i]["annotations"][0]["result"]
    
        # Step 1: Clean HTML to plain text
        soup = BeautifulSoup(raw_html, 'html.parser')
        plain_text = soup.get_text()
    
        # Step 2: Extract context snippets from clean text
        for ann in annotations:
            value = ann.get("value", {})
            if "globalOffsets" not in value or "text" not in value:
                continue
    
            label = value.get("hypertextlabels", [])
            entity_text = value["text"]
            start = value["globalOffsets"]["start"]
            end = value["globalOffsets"]["end"]
    
            context_start = max(0, start - padding)
            context_end = min(len(plain_text), end + padding)
            snippet = plain_text[context_start:context_end]
    
            snippets[data[i]['id']].append({
                "label": label,
                "text": entity_text,
                "context_snippet": snippet
            })

    return snippets

snippets = extract_clean_text_snippets(merged_, padding=100)

for doc_id, s_list in snippets.items():
    print(f'\nDoc ID: {doc_id}')
    for s in s_list[:5]:
        print(f"\nLabel: {s['label']}")
        print(f"Text: {s['text']}")
        print(f"Context Snippet:\n{s['context_snippet']}\n{'-'*40}")


Doc ID: 1

Label: ['Organization Name']
Text: FRANKLIN RESOURCES, INC.
Context Snippet:
k Constant            Name:  Mark ConstantTitle:  Vice President and Treasurer[Signature Page to Credit Agreement]BANK OF AMERICA, N.A.,as Administrative AgentBy:  /s/    Alexandra M. Knights            Name:  Alexandra M. K
----------------------------------------

Label: ['Person Name']
Text: Mark Constant
Context Snippet:
reasurer[Signature Page to Credit Agreement]BANK OF AMERICA, N.A.,as Administrative AgentBy:  /s/    Alexandra M. Knights            Name:  Alexandra M. KnightsTitle:    Authorized Signatory[Signature Page to Cred
----------------------------------------

Label: ['Person Position']
Text: Vice President
Context Snippet:
ge to Credit Agreement]BANK OF AMERICA, N.A.,as Administrative AgentBy:  /s/    Alexandra M. Knights            Name:  Alexandra M. KnightsTitle:    Authorized Signatory[Signature Page to Credit Agreement]BANK OF A
----------------------------------------

Label:

In [36]:
for doc_id, s_list in snippets.items():
    print(f'{doc_id}: {len(s_list)}')

1: 29
2: 77
3: 335
4: 71
5: 56
6: 25
7: 25
8: 21
9: 106
10: 33
11: 47
12: 65
13: 16
14: 40
15: 21
16: 30
17: 39
18: 40
19: 58
20: 15
21: 71
22: 44
23: 72
24: 59
25: 53
26: 179
27: 29
28: 85
29: 276
30: 63
31: 157
32: 237
33: 47
34: 47
35: 242
36: 23
37: 459
38: 132
39: 61
40: 55
41: 66
42: 94
43: 18
44: 35
45: 20
46: 35
47: 70
48: 20
49: 66
50: 62
51: 35
52: 74
53: 27
54: 17
55: 31
56: 54
57: 156
58: 35
59: 76
60: 81
61: 52
62: 27
63: 23
64: 42
65: 85
66: 111
67: 57
68: 80
69: 91
70: 65
71: 18
72: 30
73: 68
74: 46
75: 22
76: 36
77: 61
78: 12
79: 110
80: 59
81: 10
82: 83
83: 233
84: 138
85: 148
86: 38
87: 39
88: 22
89: 40
90: 105
91: 20
92: 69
93: 26
94: 69
95: 173
96: 63
97: 38
98: 57
99: 151
100: 97
101: 28
102: 26
103: 83
104: 117
105: 56
106: 170
107: 542
108: 149
109: 22
110: 80
111: 81
112: 40
113: 41
114: 31
115: 54
116: 30
117: 48
118: 92
119: 16
120: 109
121: 139
122: 114
123: 84
124: 61
125: 30
126: 103
127: 79
128: 41
129: 26
130: 124
131: 35
132: 72
133: 57
134: 59
135: 184


In [37]:
import json
from bs4 import BeautifulSoup

def extract_merged_snippets(data, padding=100):
    merged_snippets = {}

    for doc in data:
        doc_id = doc['id']
        merged_snippets[doc_id] = []

        raw_html = doc["data"]["html"]
        annotations = doc["annotations"][0]["result"]

        # Clean HTML to plain text
        soup = BeautifulSoup(raw_html, 'html.parser')
        plain_text = soup.get_text()

        # Collect and sort spans
        spans = []
        for ann in annotations:
            value = ann.get("value", {})
            if "globalOffsets" not in value or "text" not in value:
                continue

            start = value["globalOffsets"]["start"]
            end = value["globalOffsets"]["end"]
            spans.append((max(0, start - padding), min(len(plain_text), end + padding)))

        spans.sort()

        # Merge overlapping/adjacent spans
        merged = []
        for span in spans:
            if not merged:
                merged.append(span)
            else:
                last_start, last_end = merged[-1]
                curr_start, curr_end = span
                if curr_start <= last_end:
                    merged[-1] = (last_start, max(last_end, curr_end))
                else:
                    merged.append(span)

        # Extract text for merged spans
        for start, end in merged:
            snippet = plain_text[start:end]
            merged_snippets[doc_id].append(snippet)

    return merged_snippets

# Example usage
snippets = extract_merged_snippets(merged_, padding=100)

for doc_id, s_list in snippets.items():
    print(f'\nDoc ID: {doc_id}')
    for i, snippet in enumerate(s_list[:5], 1):
        print(f"\nSnippet {i}:\n{snippet}\n{'-'*40}")


Doc ID: 1

Snippet 1:
agreement.htm
EX-10.1




DocumentEXHIBIT 10.1CREDIT AGREEMENTdated as of January 10, 2022amongFRANKLIN RESOURCES, INC.,as the Borrower,BANK OF AMERICA, N.A.,as Administrative Agent,andthe Lenders party heretoBANK OF AMERICA, N.A.,asSole Lead Arranger and Sole Bookrunner                TABLE OF CONTENTSSectionPageARTICLE IDEFINITIONS AND ACCOUNTING TERMS11.01Defined Terms11.0
----------------------------------------

Snippet 2:
is CREDIT AGREEMENT (“Agreement”) is entered into as of January 10, 2022, among FRANKLIN RESOURCES, INC., a Delaware corporation (the “Borrower”), each lender from time to time party hereto (collectively, the “Lenders” and individually, a “Lender”), and BANK OF AMERICA, N.A., as Administrative Agent.The Borrower has requested that the Lenders provide a revolving credit facility, and the Lenders are willing to do so o
----------------------------------------

Snippet 3:
Affiliate of an entity that administers or manages a Lender.“Arranger” 

# Reorganizing

In [6]:
merged_ = read_json('final_merged.json')

In [7]:
for merged_doc in merged_:
    print(merged_doc['id'])

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170


In [51]:
merged_[0]

{'id': 1,
 'annotations': [{'id': 1,
   'completed_by': 1,
   'result': [{'value': {'start': '/document[1]/text[1]/div[886]/font[1]/text()[1]',
      'startOffset': 0,
      'end': '/document[1]/text[1]/div[886]/font[1]/text()[1]',
      'endOffset': 24,
      'globalOffsets': {'start': 305599, 'end': 305623},
      'text': 'FRANKLIN RESOURCES, INC.',
      'hypertextlabels': ['Organization Name']},
     'id': 'aaaaaaaaaa',
     'from_name': 'ner',
     'to_name': 'text',
     'type': 'hypertextlabels',
     'origin': 'manual'},
    {'value': {'start': '/document[1]/text[1]/div[890]/font[1]/text()[1]',
      'startOffset': 7,
      'end': '/document[1]/text[1]/div[890]/font[1]/text()[1]',
      'endOffset': 20,
      'globalOffsets': {'start': 305669, 'end': 305682},
      'text': 'Mark Constant',
      'hypertextlabels': ['Person Name']},
     'id': 'aaaaaaaaab',
     'from_name': 'ner',
     'to_name': 'text',
     'type': 'hypertextlabels',
     'origin': 'manual'},
    {'value': {'

In [5]:
htmls = read_html_files('./raw/')

In [6]:
mapping = {}
for i in range(len(merged_)):
    error = True
    for html_id, html_content in htmls.items():
        if merged_[i]['data']['html'] == html_content:
            if html_id[0] not in mapping:
                mapping[html_id[0]] = []
            mapping[html_id[0]].append(merged_[i]['id'])
            error = False
            break
    if error:
        print(merged_[i]['id'])

81


In [7]:
for year, ids in mapping.items():
    print(year, len(ids))

2022 17
2013 17
2014 16
2018 17
2019 17
2020 17
2021 17
2015 17
2016 17
2017 17


In [8]:
mapping[2014].append(81)

In [9]:
for year, ids in mapping.items():
    print(year, len(ids))

2022 17
2013 17
2014 17
2018 17
2019 17
2020 17
2021 17
2015 17
2016 17
2017 17


In [10]:
dev_set = []
test_set = []
proprietary_set = []
for _, doc_ids in mapping.items():
    dev_set += doc_ids[:4]
    test_set += doc_ids[4:]
    proprietary_set += doc_ids[4:5]

In [11]:
sorted(dev_set)

[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 18,
 19,
 20,
 21,
 22,
 23,
 28,
 29,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 44,
 46,
 48,
 50,
 63,
 64]

In [12]:
sorted(test_set)

[9,
 17,
 24,
 25,
 26,
 27,
 30,
 31,
 43,
 45,
 47,
 49,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170]

In [13]:
sorted(proprietary_set)

[9, 17, 25, 43, 45, 47, 51, 60, 65, 66]

In [14]:
dev_docs = []
test_docs = []
proprietary_docs = []
for i in range(len(merged_)):
    if merged_[i]['id'] in dev_set:
        dev_docs.append(merged_[i])
    else:
        test_docs.append(merged_[i])
        if merged_[i]['id'] in proprietary_set:
            proprietary_docs.append(merged_[i])

In [15]:
save_json(dev_docs, 'docs_dev.json')
save_json(test_docs, 'docs_test.json')
save_json(proprietary_docs, 'docs_proprietary.json')

# QAs

In [16]:
import pandas as pd

qa_pairs = pd.read_csv('L5.csv')
qa_pairs.info()
qa_pairs.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 450 entries, 0 to 449
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   document_number     450 non-null    int64 
 1   question            450 non-null    object
 2   answer              450 non-null    object
 3   num_hops            450 non-null    int64 
 4   num_set_operations  450 non-null    int64 
 5   multiple_answers    450 non-null    object
dtypes: int64(3), object(3)
memory usage: 21.2+ KB


,document_number,question,answer,num_hops,num_set_operations,multiple_answers
0,139,who is the senior vice president but not chief...,jay a. brown,3,2,0
1,139,who is the vice president but not authorized s...,michael king,3,2,0
2,139,who is the vice president but not authorized s...,michael king,3,2,0
3,139,who is the vice president but not authorized s...,michael king,3,2,0
4,139,who is the vice president but not authorized s...,michael king,3,2,0


In [17]:
dev_qas = qa_pairs[qa_pairs['document_number'].isin(dev_set)]
dev_qas.info()
dev_qas.head()

<class 'pandas.core.frame.DataFrame'>
Index: 157 entries, 86 to 344
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   document_number     157 non-null    int64 
 1   question            157 non-null    object
 2   answer              157 non-null    object
 3   num_hops            157 non-null    int64 
 4   num_set_operations  157 non-null    int64 
 5   multiple_answers    157 non-null    object
dtypes: int64(3), object(3)
memory usage: 8.6+ KB


,document_number,question,answer,num_hops,num_set_operations,multiple_answers
86,11,who is the vice president but not treasurer of...,john j. tus,3,2,0
87,11,who is the vice president but not treasurer of...,john j. tus,3,2,0
90,38,who is the executive director of the company w...,dave katz,3,2,0
123,41,who is the director of the company which is th...,john dillon,3,2,0
124,41,who is the director of the company which is th...,john dillon,3,2,0


In [18]:
test_qas = qa_pairs[qa_pairs['document_number'].isin(test_set)]
test_qas.info()
test_qas.head()

<class 'pandas.core.frame.DataFrame'>
Index: 293 entries, 0 to 449
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   document_number     293 non-null    int64 
 1   question            293 non-null    object
 2   answer              293 non-null    object
 3   num_hops            293 non-null    int64 
 4   num_set_operations  293 non-null    int64 
 5   multiple_answers    293 non-null    object
dtypes: int64(3), object(3)
memory usage: 16.0+ KB


,document_number,question,answer,num_hops,num_set_operations,multiple_answers
0,139,who is the senior vice president but not chief...,jay a. brown,3,2,0
1,139,who is the vice president but not authorized s...,michael king,3,2,0
2,139,who is the vice president but not authorized s...,michael king,3,2,0
3,139,who is the vice president but not authorized s...,michael king,3,2,0
4,139,who is the vice president but not authorized s...,michael king,3,2,0


In [19]:
the_rest = qa_pairs[(~qa_pairs['document_number'].isin(test_set)) & (~qa_pairs['document_number'].isin(dev_set))]
the_rest.info()
the_rest.head()

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   document_number     0 non-null      int64 
 1   question            0 non-null      object
 2   answer              0 non-null      object
 3   num_hops            0 non-null      int64 
 4   num_set_operations  0 non-null      int64 
 5   multiple_answers    0 non-null      object
dtypes: int64(3), object(3)
memory usage: 0.0+ bytes


,document_number,question,answer,num_hops,num_set_operations,multiple_answers


In [20]:
proprietary_qas = qa_pairs[qa_pairs['document_number'].isin(proprietary_set)]
proprietary_qas.info()
proprietary_qas.head()

<class 'pandas.core.frame.DataFrame'>
Index: 101 entries, 112 to 434
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   document_number     101 non-null    int64 
 1   question            101 non-null    object
 2   answer              101 non-null    object
 3   num_hops            101 non-null    int64 
 4   num_set_operations  101 non-null    int64 
 5   multiple_answers    101 non-null    object
dtypes: int64(3), object(3)
memory usage: 5.5+ KB


,document_number,question,answer,num_hops,num_set_operations,multiple_answers
112,66,who is the authorized signer of the company wh...,james riley,3,2,0
113,66,who is the authorized signer of the company wh...,james riley,3,2,0
114,66,who is the authorized signer of the company wh...,james riley,3,2,0
116,9,who is the senior vice president but not chief...,mark e. newman,3,2,0
117,9,who is the senior vice president but not chief...,mark e. newman,3,2,0


### Test

In [21]:
test_qas['document_number'].value_counts().to_dict()

{60: 71,
 166: 15,
 135: 15,
 107: 14,
 66: 13,
 94: 11,
 121: 10,
 83: 8,
 151: 8,
 147: 7,
 51: 6,
 96: 6,
 130: 6,
 169: 6,
 61: 5,
 47: 5,
 84: 5,
 139: 5,
 145: 5,
 153: 5,
 77: 4,
 162: 4,
 65: 4,
 68: 4,
 115: 4,
 86: 3,
 142: 3,
 59: 3,
 137: 3,
 124: 2,
 87: 2,
 90: 2,
 72: 2,
 128: 2,
 156: 2,
 9: 2,
 165: 2,
 140: 2,
 155: 2,
 106: 2,
 111: 2,
 144: 2,
 26: 1,
 127: 1,
 58: 1,
 54: 1,
 112: 1,
 89: 1,
 160: 1,
 99: 1,
 126: 1}

In [22]:
category_counts = test_qas['document_number'].value_counts()
valid_categories = category_counts[category_counts > 4].index
test_qas = test_qas[test_qas['document_number'].isin(valid_categories)]

In [23]:
test_qas['document_number'].value_counts().to_dict()

{60: 71,
 135: 15,
 166: 15,
 107: 14,
 66: 13,
 94: 11,
 121: 10,
 151: 8,
 83: 8,
 147: 7,
 169: 6,
 96: 6,
 51: 6,
 130: 6,
 145: 5,
 153: 5,
 61: 5,
 47: 5,
 84: 5,
 139: 5}

### Dev

In [24]:
dev_qas['document_number'].value_counts().to_dict()

{32: 117,
 35: 13,
 40: 8,
 22: 5,
 41: 3,
 11: 2,
 5: 2,
 23: 2,
 38: 1,
 42: 1,
 46: 1,
 3: 1,
 37: 1}

In [25]:
category_counts = dev_qas['document_number'].value_counts()
valid_categories = category_counts[category_counts > 4].index
dev_qas = dev_qas[dev_qas['document_number'].isin(valid_categories)]

In [26]:
dev_qas['document_number'].value_counts().to_dict()

{32: 117, 35: 13, 40: 8, 22: 5}

### Prop

In [27]:
proprietary_qas['document_number'].value_counts().to_dict()

{60: 71, 66: 13, 51: 6, 47: 5, 65: 4, 9: 2}

In [28]:
category_counts = proprietary_qas['document_number'].value_counts()
valid_categories = category_counts[category_counts > 4].index
proprietary_qas = proprietary_qas[proprietary_qas['document_number'].isin(valid_categories)]

In [29]:
proprietary_qas['document_number'].value_counts().to_dict()

{60: 71, 66: 13, 51: 6, 47: 5}

In [30]:
dev_qas.to_csv('L5_dev.csv')

In [31]:
test_qas.to_csv('L5_test.csv')

In [32]:
proprietary_qas.to_csv('L5_proprietary.csv')